### Set Up

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
from scipy.stats import chi2_contingency, ttest_ind

### Define Functions 

In [2]:
#  takes in an annotator output and returns a dictionary with the events as keys and the binary labellings as values

def parse_events_with_labels(file_path):
    """
    Parse a JSON file containing event nodes and extract their C/I/K polarity labels.
    
    Args:
        file_path: Path to the JSON file
        
    Returns:
        dict: Mapping of event labels to their C/I/K polarity strings
              e.g., {"Historical buildings are demolished": "C+I+K+", ...}
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Parse each line as a separate JSON object
    nodes = [json.loads(line.strip()) for line in lines if line.strip()]
    
    # Find the being node (first node with kind "being")
    being_node = None
    for node_obj in nodes:
        if node_obj.get('node', {}).get('kind') == 'being':
            being_node = node_obj
            break
    
    if not being_node:
        return {}
                    
    
    # Create a mapping from event labels to their C/I/K values
    event_labels = {}
    
    for link in being_node.get('links', []):
        to_node_label = link.get('to_node')
        b_link_value = link.get('link', {}).get('value')
        
        if to_node_label and b_link_value:
            event_labels[to_node_label] = [b_link_value]
    
    # Filter to only include actual events
    event_nodes = {
        node_obj['node']['label'] 
        for node_obj in nodes 
        if node_obj.get('node', {}).get('kind') == 'event'
    }

    # attach the utility value to the event labels
    for node_obj in nodes:
        if node_obj.get('node', {}).get('kind') == 'event':
            event_label = node_obj['node']['label']
            for link in node_obj.get('links', []):
                if link.get('to_node') == being_node.get('node', {}).get('label') and link.get('link', {}).get('kind') == 'utility':
                    utility_value = link.get('link', {}).get('value')
                    event_labels[event_label].append(utility_value)

    
    return {label: value for label, value in event_labels.items() if label in event_nodes}

## Process Mild Harm Scenarios -- THE Analysis Starting Point

In [12]:
#select the annotated scenario outputs that will be analyzed
mild_annotated_output_path = Path().resolve() / "../../annotated_outputs/franken/conditions_mild_harm_mild_good"
print(f"mild_annotated_output_path: {mild_annotated_output_path}")

mild_annotated_output_path: /home/bhatto/graph_extract/analysis/franken/../../annotated_outputs/franken/conditions_mild_harm_mild_good


In [ ]:
#  create dataframe with all annotator-generated events for choice 1s of all scenarios in this set

mild_annotator_inputs_megadf = []
for folder in mild_annotated_output_path.iterdir():
    if folder.is_dir():
        evitability = "Inevitable" if "inevitable" in folder.name else "Evitable"
        means_side_effect = "CC (Means)" if "cc" in folder.name else "COC (SideEff)"
        co_omission = "Commission" if "action_yes" in folder.name else "Omission"
        
        for json_file in folder.glob("*choice_1.json"):
            sid = json_file.stem.split("_")[0]  # Extract SID from filename
            
            event_labels = parse_events_with_labels(json_file)
            
            for event, cik_value in event_labels.items():
                c_value = cik_value[0][1]  # C polarity
                i_value = cik_value[0][3]  # I polarity
                k_value = cik_value[0][5]  # K polarity
                event_utility = cik_value[1]  # Utility value
                
                mild_annotator_inputs_megadf.append({
                    "SID": sid,
                    "Folder name": folder.name,
                    "Evitability": evitability,
                    "Means/Side Effect": means_side_effect,
                    "Commission/Omission": co_omission,
                    "Event": event,
                    "C": c_value,
                    "I": i_value,
                    "K": k_value,
                    "utility": event_utility  # Utility value

                })

folder_order = [
    "cc_evitable_action_yes_stories",
    "cc_evitable_prevention_no_stories",
    "cc_inevitable_action_yes_stories",
    "cc_inevitable_prevention_no_stories",
    "coc_evitable_action_yes_stories",
    "coc_evitable_prevention_no_stories",
    "coc_inevitable_action_yes_stories",
    "coc_inevitable_prevention_no_stories"
]

# reorder the rows that the SID order is ascending within each folder
mild_annotator_inputs_megadf.sort(key=lambda x: (folder_order.index(x["Folder name"]), int(x["SID"])))

print(f'preview of all mild harm/mild good events dataframe with annotator-generated C/I/K labels for each event:')
display(HTML(pd.DataFrame(mild_annotator_inputs_megadf)[:10].to_html()))
print(f'total number of events in the mild harm/mild good events dataframe: {len(mild_annotator_inputs_megadf)}')

preview of all mild harm/mild good events dataframe with annotator-generated C/I/K labels for each event:


,SID,Folder name,Evitability,Means/Side Effect,Commission/Omission,Event,C,I,K,utility
0,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,I initiate renovation work in the park,+,+,+,20
1,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The community temporarily loses access to the recreational space,+,+,+,-10
2,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Residents experience inconvenience due to park closure,+,+,+,-20
3,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Residents feel disappointment over the temporary loss of the park,+,+,+,-20
4,0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The park's facilities are improved after renovation,+,+,+,80
5,1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,I initiate the server system upgrade,+,+,+,20
6,1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The company's server system goes offline for a few hours,+,+,+,-20
7,1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The company's operations temporarily halt during the server downtime,+,+,+,-20
8,1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Data processing speed of the company's server system improves after the upgrade,+,+,-,70
9,1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Employees experience a temporary disruption in their work,+,+,+,-20


total number of events in the mild harm/mild good events dataframe: 1198


In [14]:
# ADD HERE: SELECTION OF PRIMARY HARMS

# Human Ratings Data

## Experiment 2 is relevant: human likert ratings on moral permissibility (ignore for now) and intention (aka our intention) on scale of 1-5
### Problem -- they only ran a subset of mild scenarios

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# load exp2 human data:
human_ratings = pd.read_csv(Path().resolve() / "../../human_data/franken/exp2_moralperm-intent-rating/data_long_format.csv")
human_ratings = human_ratings.drop(columns=['scenario_harm', 'split'])
print("preview of raw exp2 human ratings data:\n")
display(HTML(human_ratings[0:10].to_html())) # preview
# count number of unique combinations of scenario_id + causal_structure + evitability + action
exp2_inputs_df = human_ratings.groupby(['scenario_id', 'causal_structure', 'evitability', 'action']).size().reset_index(name='counts') # each scenario got rated by ~20-25 participants
# add a column of average rating of moral permissibility and intention for each unique combination
avg_ratings = human_ratings.groupby(['scenario_id', 'causal_structure', 'evitability', 'action'])[['permissibility_rating', 'intention_rating']].mean().reset_index()
exp2_inputs_df = exp2_inputs_df.merge(avg_ratings, on=['scenario_id', 'causal_structure', 'evitability', 'action'])
exp2_inputs_df = exp2_inputs_df.rename(columns={
    'permissibility_rating': 'avg_permissibility_rating',
    'intention_rating': 'avg_intention_rating'
})
print("preview of all exp2 human ratings, grouped by scenario_id, causal_structure, evitability, and action, and averaged ratings:")
print("I call this the exp2_inputs_df dataframe and each row is a different scenario:")
print("\n Explainer: \n causal_structure (0 for means (cc), 1 for side effect (coc)) \n action (0 for commission, 1 for omission (prevention))  \n evitability (0 for evitable, 1 for inevitable) \n avg_permissibility_rating (average moral permissibility rating for that scenario) \n avg_intention_rating (average intention rating for that scenario)")
display(HTML(exp2_inputs_df[:20].to_html()))
print(f"Total number of scenarios rated: {len(exp2_inputs_df)}")

# Causal structure (0 for means (cc), 1 for side effect (coc))

# Action (0 for commission, 1 for omission (prevention))

# Evitability (0 for evitable, 1 for inevitable)

# ** only 'mild' scenarios are used in this experiment **

preview of raw exp2 human ratings data:



,worker_id,scenario_id,permissibility_rating,intention_rating,causal_structure,evitability,action
0,226,5,3,3,1,0,1
1,226,9,5,2,0,1,1
2,226,4,3,3,0,1,1
3,226,2,3,3,1,0,0
4,226,5,4,2,0,1,1
5,226,6,4,2,1,0,1
6,226,1,5,2,1,1,0
7,226,0,4,3,1,0,0
8,226,3,4,3,0,1,0
9,226,2,5,2,0,1,1


preview of all exp2 human ratings, grouped by scenario_id, causal_structure, evitability, and action, and averaged ratings:
I call this the unique_combinations dataframe and each row is a different scenario:

 Explainer: 
 causal_structure (0 for means (cc), 1 for side effect (coc)) 
 action (0 for commission, 1 for omission (prevention))  
 evitability (0 for evitable, 1 for inevitable) 
 avg_permissibility_rating (average moral permissibility rating for that scenario) 
 avg_intention_rating (average intention rating for that scenario)


,scenario_id,causal_structure,evitability,action,counts,avg_permissibility_rating,avg_intention_rating
0,0,0,0,0,26,4.153846,1.923077
1,0,0,0,1,25,4.280000,1.960000
2,0,0,1,0,21,3.952381,2.476190
3,0,0,1,1,21,3.952381,2.666667
4,0,1,0,0,22,4.272727,2.045455
5,0,1,0,1,21,4.238095,1.952381
6,0,1,1,0,22,4.045455,2.409091
7,0,1,1,1,26,3.961538,2.269231
8,1,0,0,0,21,4.095238,2.095238
9,1,0,0,1,26,4.384615,1.923077


Total number of scenarios rated: 80


## Exp2 - Pick out all the annotated output scenario rows for which human ratings exist

In [ ]:
# Go through the exp2_inputs_df df. For each unique combination of scenario_id + causal_structure + evitability + action, find the corresponding row in the data dataframe. 
# Do this by finding the correct folder -- "cc_evitable_action_yes_stories" would be the correct folder for the combination of causal_structure = 0 (means/cc), evitability = 0 (evitable), action = 0 (commission/action_yes). 
# Then, within that folder, find the row with the correct scenario_id (SID) and extract the events and its C/I/K values. Do this for all rows in exp2_inputs_df, and create a new dataframe with only the annotator-generated events (and their C/I/K labels) for the scenarios that were rated in exp2.:

annotator_megadf_filtered_for_exp2_inputs = []
for _, row in exp2_inputs_df.iterrows():
    scenario_id = int(row['scenario_id'])
    causal_structure = row['causal_structure']
    evitability = row['evitability']
    action = row['action']
    
    # Determine folder name based on combination
    folder_name = f"{'cc' if causal_structure == 0 else 'coc'}_{'evitable' if evitability == 0 else 'inevitable'}_{'action_yes' if action == 0 else 'prevention_no'}_stories"
    
    # Find corresponding rows in data dataframe
    matching_rows = [d for d in mild_annotator_inputs_megadf if d['Folder name'] == folder_name and d['SID'] == str(scenario_id)]
    # add the row index from the exp2_inputs_df dataframe to each of the matching rows
    for mr in matching_rows:
        mr['exp2_inputs_df_index'] = row.name  # row.name gives the index of the row in exp2_inputs_df dataframe
    
    annotator_megadf_filtered_for_exp2_inputs.extend(matching_rows)
annotator_megadf_filtered_for_exp2_inputs = pd.DataFrame(annotator_megadf_filtered_for_exp2_inputs)
annotator_megadf_filtered_for_exp2_inputs = annotator_megadf_filtered_for_exp2_inputs.drop(columns=['SID'])
print("length of full mild harm/good event dataframe:", len(mild_annotator_inputs_megadf))
print("length of annotator dataframe after removing everything outside of the exp2_inputs_df:", len(annotator_megadf_filtered_for_exp2_inputs))
print("preview of exp2-filtered annotator df (and their C/I/K labels):\n")
print("these are annotator-generated events (and their labels) for all the scenarios that were rated in exp2.\n")
print("each scenario got 5-6 generated events, and these are the events are for scenarios that the human ratings in exp2 correspond to.")
display(HTML(annotator_megadf_filtered_for_exp2_inputs[:20].to_html()))
# display(HTML(annotator_megadf_filtered_for_exp2_inputs.to_html()))
# print(annotator_megadf_filtered_for_exp2_inputs.head())

length of full mild harm/good event dataframe: 1198
length of human-rated event dataframe: 409
preview of annotator-generated events (and their C/I/K labels) for the human-rated scenarios from exp2:



,Folder name,Evitability,Means/Side Effect,Commission/Omission,Event,C,I,K,utility,unique_combination_index
0,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,I initiate renovation work in the park,+,+,+,20,0
1,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The community temporarily loses access to the recreational space,+,+,+,-10,0
2,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Residents experience inconvenience due to park closure,+,+,+,-20,0
3,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,Residents feel disappointment over the temporary loss of the park,+,+,+,-20,0
4,cc_evitable_action_yes_stories,Evitable,CC (Means),Commission,The park's facilities are improved after renovation,+,+,+,80,0
5,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,The community retains continuous access to the beloved recreational space.,+,+,+,50,1
6,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,Residents do not experience temporary inconvenience due to park closure.,+,+,+,20,1
7,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,The park facilities remain in their current state without improvements.,+,+,+,20,1
8,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,The colleague's opportunity to renovate the park is blocked.,+,+,+,10,1
9,cc_evitable_prevention_no_stories,Evitable,CC (Means),Omission,Potential long-term benefits from park improvements are not realized.,+,-,+,-20,1


### **NOTE:** The right-most column tells you which human-rated scenario these events were generated for. Goes from 0-79.
### So the annotator gives us labels for the multiple events that come out of a scenario. And the human ratings are for the scenarios themselves. 
### We'll need to isolate one event for each scenario.

In [ ]:
# pick out the primary harm event for each human-rated scenario in exp2
primary_harm_rows_exp2 = [2, 9, 12, 21, 25, 34, 39, 49, 53, 61,                # primary harm for scenarios 1-10
                          64, 69, 76, 78, 85, 91, 93, 100, 103, 107,           # primary harm for scenarios 11-20
                          109, 113, 117, 123, 126, 131, 135, 141, 144, 148]    # primary harm for scenarios 21-30

In [ ]:
# filter out only the primary harm events for the human-rated scenarios in exp2 into a new df
primary_harms_exp2_filtered_megadf = annotator_megadf_filtered_for_exp2_inputs.iloc[primary_harm_rows_exp2]
# attach the avg_intention_rating and avg_permissibility_rating from the exp2_inputs_df df to the primary_harm_events_exp2_df from the first 30 rows of exp2_inputs_df:
ratings_to_attach = exp2_inputs_df[['avg_intention_rating', 'avg_permissibility_rating']].iloc[0:30].reset_index(drop=True)
primary_harms_exp2_filtered_megadf = primary_harms_exp2_filtered_megadf.reset_index(drop=True)
primary_harms_exp2_filtered_megadf = pd.concat([primary_harms_exp2_filtered_megadf, ratings_to_attach], axis=1)
print("dataframe of primary harm events for the human-rated scenarios in exp2:\n")
display(HTML(primary_harms_exp2_filtered_megadf.to_html(index=False)))

# ----------------------------------------------------- EXP 1 STUFF -----------------------------------------------------

In [17]:
# Load exp1 data:
exp1 = pd.read_csv(Path().resolve() / "../../human_data/franken/exp1_harm-vs-good-rating/data_combined.csv", delimiter='\t')
exp1 = exp1.drop(columns=['proliferate.condition', 'error'])
print("preview of raw exp1 human ratings data:\n")
display(HTML(exp1[0:1].to_html())) # preview

preview of raw exp1 human ratings data:



,workerid,trial1,trial10,trial11,trial12,trial13,trial14,trial15,trial16,trial17,trial18,trial19,trial2,trial20,trial21,trial22,trial23,trial24,trial25,trial26,trial27,trial28,trial29,trial3,trial30,trial31,trial32,trial33,trial34,trial35,trial36,trial37,trial38,trial39,trial4,trial40,trial5,trial6,trial7,trial8,trial9
0,58,"{'type': 'good', 'target': 'The new bike lane system encourages residents to cycle more.', 'strength': 'mild', 'structure': 'coc', 'background': 'Maya has the opportunity to implement a new bike lane system throughout the city.', 'scenario_id': 0, 'likertResponses': {'likert1': '7'}}","{'type': 'harm', 'target': ""This implementation requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days."", 'strength': 'severe', 'structure': 'cc', 'background': 'Alejandro has the opportunity to implement a new power grid system in the city.', 'scenario_id': 1, 'likertResponses': {'likert1': '2'}}","{'type': 'harm', 'target': 'The implementation of the new scheduling system requires a temporary disruption in booking appointments, causing a delay for patients needing to schedule their visits.', 'strength': 'mild', 'structure': 'cc', 'background': 'Mei has the opportunity to implement a new scheduling system for patient appointments.', 'scenario_id': 3, 'likertResponses': {'likert1': '3'}}","{'type': 'harm', 'target': ""The change in menu requires removing some of the customer's favorite dishes, causing disappointment among regular customers."", 'strength': 'mild', 'structure': 'cc', 'background': 'Amina has the opportunity to change the menu of her restaurant.', 'scenario_id': 4, 'likertResponses': {'likert1': '3'}}","{'type': 'harm', 'target': 'Some elderly patients feel uncomfortable with the new system, missing the personal touch of traditional methods.', 'strength': 'mild', 'structure': 'coc', 'background': 'Mei has the opportunity to implement a new digital system for patient record-keeping in her clinic.', 'scenario_id': 3, 'likertResponses': {'likert1': '4'}}","{'type': 'good', 'target': ""The new experimental drug has the potential to completely cure the patient's illness, leading to a full recovery and a healthy life."", 'strength': 'severe', 'structure': 'cc', 'background': 'Mei has the opportunity to administer a new experimental drug to a critically ill patient.', 'scenario_id': 3, 'likertResponses': {'likert1': '7'}}","{'type': 'harm', 'target': 'This renovation requires temporarily deprive the community of a beloved recreational space, causing inconvenience and disappointment among residents.', 'strength': 'mild', 'structure': 'cc', 'background': 'Maya has the opportunity to renovate a park in the city.', 'scenario_id': 0, 'likertResponses': {'likert1': '3'}}","{'type': 'harm', 'target': 'Some students who are less tech-savvy experience initial confusion and anxiety adapting to the new system.', 'strength': 'mild', 'structure': 'coc', 'background': 'Amir has the opportunity to introduce an online homework submission system in his class.', 'scenario_id': 2, 'likertResponses': {'likert1': '3'}}","{'type': 'good', 'target': ""The experimental drug has the potential to save the patient's life, who is otherwise not responding to any available treatments."", 'strength': 'severe', 'structure': 'coc', 'background': 'Mei has the opportunity to administer an experimental drug to a critically ill patient.', 'scenario_id': 3, 'likertResponses': {'likert1': '7'}}","{'type': 'good', 'target': 'The upgraded server system will slightly improve data processing speed.', 'strength': 'mild', 'structure': 'cc', 'background': ""Alejandro has the opportunity to upgrade the company's server system."", 'scenario_id': 1, 'likertResponses': {'likert1': '6'}}","{'type': 'harm', 'target': 'The new microwave is slightly noisier, causing disturbance to employees sitting nearby.', 'strength': 'mild', 'structure': 'coc', 'background': 'Alejandro has the opportunity to upgrade a microwave to a m

### They provide human raters with a scenario 'background' AND a 'target' sentence which describes the primary harm and asks raters to rate moral 'goodness' vs harm on a 1-7 scale (aka our utility)

## Load Exp 1 processed version -- already picked out unique inputs and calculated average human likert rating for each

In [37]:
exp1_processed = pd.read_json(Path().resolve() / "../../human_data/franken/exp1_harm-vs-good-rating/exp1_unique_stimuli.json")
# name the index of exp1_processed "stimulus_id"
exp1_processed.index.name = "exp1_input_id"
print("preview of processed exp1 unique stimuli data:\n")
display(HTML(exp1_processed[:35].to_html()))
print(f"Total number of unique stimuli in exp1: {len(exp1_processed)}")

preview of processed exp1 unique stimuli data:



,type,target,strength,structure,background,avg_likert_rating,n_likert_ratings
exp1_input_id,,,,,,,
0,good,The new bike lane system encourages residents to cycle more.,mild,coc,Maya has the opportunity to implement a new bike lane system throughout the city.,6.05,20
1,harm,"This implementation requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days.",severe,cc,Alejandro has the opportunity to implement a new power grid system in the city.,2.10,20
2,harm,"The implementation of the new scheduling system requires a temporary disruption in booking appointments, causing a delay for patients needing to schedule their visits.",mild,cc,Mei has the opportunity to implement a new scheduling system for patient appointments.,3.25,20
3,harm,"The change in menu requires removing some of the customer's favorite dishes, causing disappointment among regular customers.",mild,cc,Amina has the opportunity to change the menu of her restaurant.,3.05,20
4,harm,"Some elderly patients feel uncomfortable with the new system, missing the personal touch of traditional methods.",mild,coc,Mei has the opportunity to implement a new digital system for patient record-keeping in her clinic.,3.60,20
5,good,"The new experimental drug has the potential to completely cure the patient's illness, leading to a full recovery and a healthy life.",severe,cc,Mei has the opportunity to administer a new experimental drug to a critically ill patient.,6.40,20
6,harm,"This renovation requires temporarily deprive the community of a beloved recreational space, causing inconvenience and disappointment among residents.",mild,cc,Maya has the opportunity to renovate a park in the city.,3.45,20
7,harm,Some students who are less tech-savvy experience initial confusion and anxiety adapting to the new system.,mild,coc,Amir has the opportunity to introduce an online homework submission system in his class.,3.30,20
8,good,"The experimental drug has the potential to save the patient's life, who is otherwise not responding to any available treatments.",severe,coc,Mei has the opportunity to administer an experimental drug to a critically ill patient.,6.00,20


Total number of unique stimuli in exp1: 80


## Exp1 -- trying to match up experiment stimuli to the granular scenarios that were fed to the annotator (so we can then extract annotator ratings for its events)

In [ ]:
# loop through exp1_processed. Each row is indexed with a 'exp1_input_id'. For each row, look at the strength and structure column values, and also extract the first word from the background column value, which will always be a person's name. Then, look at the csv called "namedinputs_{strength}_{structure}.csv in ../../scenarios_inputs/franken/". In this CSV, collect all the rows where the first word of the first item is also the same person's name. For each such row, take all the column entries in that row and join them into one string value separated by spaces. Create a new dataframe with the exp1_input_id value, the strength and structure column values, the background column value, and a new column called 'potential_match_scenarios' which contains a disctionary where the keys are the row numbers of the matching rows in the namedinputs csv and the values are the joined string of all column entries for that row. Display this dataframe. This will show, for each unique stimulus in exp1, which scenarios from the scenario inputs have the same person's name in them and could be potential matches for that stimulus.
potential_matches = []
for idx, row in exp1_processed.iterrows():
    exp1_input_id = idx
    strength = row['strength']
    structure = row['structure']
    background = row['background']
    target = row['target']
    type = row['type']
    first_word = background.split()[0]  # extract the first word from the background column value
    # print(f"found name: {first_word} in exp1 stimulus with id {exp1_input_id}, strength {strength}, structure {structure}")  # debug print to check the extracted name and corresponding stimulus info  
    
    # Load the corresponding namedinputs csv based on strength and structure
    namedinputs_csv_path = Path().resolve() / f"../../scenarios_inputs/franken/namedinputs_{strength}_{structure}.csv"
    namedinputs_df = pd.read_csv(namedinputs_csv_path, header=None) 
    
    # Find rows where the first word of the first item matches the extracted first word
    matching_rows = {}
    for i, named_row in namedinputs_df.iterrows():
        # print(f"named_row: {named_row}")  # debug print to check the content of named_row
        first_item_first_word = (str(named_row[0]).split()[0])[:-1]  # get the first word of the first item in the row
        if first_item_first_word == first_word:
            joined_string = ' '.join(str(x) for x in named_row)  # join all column entries into one string
            matching_rows[i] = joined_string  # use row number as key and joined string as value
    
    potential_matches.append({
        'exp1_input_id': exp1_input_id,
        # 'strength': strength,
        # 'structure': structure,
        'background': background,
        'target': target,
        'type': type,
        'potential_match_scenarios': matching_rows, 
        'num_potential_matches': len(matching_rows)
    })
    
# Create a new dataframe from the potential_matches list
potential_matches_df = pd.DataFrame(potential_matches)
# preview the df such that the doctionary in the 'potential_match_scenarios' column is pretty printed
display(HTML(potential_matches_df[0:20].to_html()))
# save the potential_matches_df as a csv file
potential_matches_df.to_csv(Path().resolve() / "../../human_data/franken/exp1_harm-vs-good-rating/exp1_potential_matches.csv", index=False)


In [20]:
exp1inputid_to_matchedkeyfrompotentialscenarios = {
    0: 0,
    1: 11,
    2: 53,
    3: 54,
    4: 3,
    5: 13,
    6: 50,
    7: 2,
    8: 3,
    9: 51,
    10: 1,
    11: 4,
    12: 2,
    13: 0,
    14: 3,
    15: 51,
    16: 11,
    17: 4,
    18: 1,
    19: 12,
}

In [21]:
# make a new df which is a copy of the exp1_processed df but with an additional column called 'matched_scenario_info' which contains the single key-value pair from the 'potential_match_scenarios' dictionary where the key is the value from the exp1inputid_to_matchedkeyfrompotentialscenarios dictionary for that row's exp1_input_id value. This will show, for each unique stimulus in exp1, which single scenario from the scenario inputs is the best match for that stimulus based on having the same person's name in it and being selected by the mapping in exp1inputid_to_matchedkeyfrompotentialscenarios.
exp1_matched_df = exp1_processed.copy()[0:20]  # only take the first 20 rows for now to check
matched_scenario_info = []
for idx, row in exp1_matched_df.iterrows():
    exp1_input_id = idx
    potential_scenarios = potential_matches_df.loc[potential_matches_df['exp1_input_id'] == exp1_input_id, 'potential_match_scenarios'].values[0]
    matched_key = exp1inputid_to_matchedkeyfrompotentialscenarios.get(exp1_input_id)
    matched_info = potential_scenarios.get(matched_key, "No match found")
    matched_scenario_info.append({matched_key: matched_info})
exp1_matched_df['matched_scenario_info'] = matched_scenario_info
# preview the new df with the matched scenario info
display(HTML(exp1_matched_df.to_html()))

,type,target,strength,structure,background,avg_likert_rating,n_likert_ratings,matched_scenario_info
exp1_input_id,,,,,,,,
0,good,The new bike lane system encourages residents to cycle more.,mild,coc,Maya has the opportunity to implement a new bike lane system throughout the city.,6.05,20,"{0: 'Maya, an Architect, faces a moral dilemma. Maya has the opportunity to implement a new bike lane system throughout the city. The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city. If Maya does not implement the bike lane system, the increase in parking prices would not happen. Maya implements the new bike lane system.'}"
1,harm,"This implementation requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days.",severe,cc,Alejandro has the opportunity to implement a new power grid system in the city.,2.10,20,"{11: 'Alejandro, an Engineer, faces a moral dilemma. Alejandro has the opportunity to implement a new power grid system in the city. As a means to achieve the significant reduction in energy consumption and energy taxes for residents, as well as promoting stable and sustainable power supply for the future, implementing a new power grid system requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days. If Alejandro does not implement the new power grid system, the city-wide blackout for several days would not happen. Alejandro implements the new power grid system.'}"
2,harm,"The implementation of the new scheduling system requires a temporary disruption in booking appointments, causing a delay for patients needing to schedule their visits.",mild,cc,Mei has the opportunity to implement a new scheduling system for patient appointments.,3.25,20,"{53: 'Mei, a Nurse, faces a moral dilemma. Mei has the opportunity to implement a new scheduling system for patient appointments. As a means to achieve a nicer looking user interface, the implementation of the new scheduling system requires a temporary disruption in booking appointments, causing a delay for patients needing to schedule their visits. If Mei does not implement the new scheduling system, the temporary disruption in booking appointments and the resulting delay for patients would not happen. Mei implements the new scheduling system.'}"
3,harm,"The change in menu requires removing some of the customer's favorite dishes, causing disappointment among regular customers.",mild,cc,Amina has the opportunity to change the menu of her restaurant.,3.05,20,"{54: 'Amina, a Chef, faces a moral dilemma. Amina has the opportunity to change the menu of her restaurant. As a means to introduce two new food options and attract a few new customers, the change in menu requires removing some of the customer's favorite dishes, causing disappointment among regular customers. If Amina does not change the menu, the removal of some of the customer's favorite dishes and the resulting disappointment would not happen. Amina changes the menu.'}"
4,harm,"Some elderly patients feel uncomfortable with the new system, missing the personal touch of traditional methods.",mild,coc,Mei has the opportunity to implement a new digital system for patient record-keeping in her clinic.,3.60,20,"{3: 'Mei, a Nurse, faces a moral dilemma. Mei has the opportunity to implement a new digital system for patient record-keeping in her clinic. The digital system improves the time it takes to access patient records. As a side effect of improving the time it takes to access patient records, some elderly patients feel uncomfortable with the new system, missing the personal touch of traditional methods. If Mei does not implement the new digital system for patient record-keeping, the discomfort of elderly patients would not happen. Mei implements the new digital system for patient record-keeping.'}"
5,good,"Th

In [22]:
# map out the range of inputid number to refer to the correct json file to pick out the correct annotator-fed input scenario, because we had fed in non-named versions of these scenarios that were specifically formatted for the annotator. Firstly, if the strength is 'mild', look in the ../../scenarios_inputs/franken/conditions_mild_harm_mild_good/ folder. Then, the structure and inputid number tells us which json file to look at and which specific json to pull out of it. If the structure is 'cc' and the inputid is between 0 and 49, look in the cc_evitable_action_yes_stories.json and pull out the json with the same "id" value as the inputid. If the structure is 'cc' and the inputid is between 50 and 99, look in the cc_evitable_prevention_no_stories.json and pull out the json with the same "id" value as the inputid minus 50. If the structure is 'cc' and the inputid is between 100 and 149, look in the cc_inevitable_action_yes_stories.json and pull out the json with the same "id" value as the inputid minus 100. If the structure is 'cc' and the inputid is between 150 and 199, look in the cc_inevitable_prevention_no_stories.json and pull out the json with the same "id" value as the inputid minus 150. same exactly for the 'coc' structure but look in the coc_... json files instead. This mapping will allow us to connect each unique stimulus in exp1 to the correct annotator-fed input scenario and its corresponding annotator-generated events and C/I/K labels.
def map_inputid_to_json_info(strength, structure, inputid):
    folder = "conditions_mild_harm_mild_good" if strength == "mild" else "conditions_severe_harm_very_good"
    base_path = Path().resolve() / f"../../scenarios_inputs/franken/{folder}"

    if strength == 'mild':
    
        if structure == 'cc':
            if 0 <= inputid < 50:
                json_file = base_path / "cc_evitable_action_yes_stories.json"
                internal_id_to_find = inputid
            elif 50 <= inputid < 100:
                json_file = base_path / "cc_evitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 50
            elif 100 <= inputid < 150:
                json_file = base_path / "cc_inevitable_action_yes_stories.json"
                internal_id_to_find = inputid - 100
            elif 150 <= inputid < 200:
                json_file = base_path / "cc_inevitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 150
            else:
                return "Invalid inputid for cc structure"
        elif structure == 'coc':
            if 0 <= inputid < 50:
                json_file = base_path / "coc_evitable_action_yes_stories.json"
                internal_id_to_find = inputid
            elif 50 <= inputid < 100:
                json_file = base_path / "coc_evitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 50
            elif 100 <= inputid < 150:
                json_file = base_path / "coc_inevitable_action_yes_stories.json"
                internal_id_to_find = inputid - 100
            elif 150 <= inputid < 200:
                json_file = base_path / "coc_inevitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 150
            else:
                return "Invalid inputid for coc structure"
        else:
            return "Invalid structure"
        
        # Load the JSON file, which contains a list of JSON objects, and find the entry with the matching id:
        with open(json_file, 'r') as f:
            inp_jsons = json.load(f)
        for j in inp_jsons:
            if j.get('id') == internal_id_to_find:
                return j, json_file.name, internal_id_to_find
        return "No matching entry found in JSON"
    
    elif strength == 'severe':
    
        if structure == 'cc':
            if 0 <= inputid < 10:
                json_file = base_path / "cc_evitable_action_yes_stories.json"
                internal_id_to_find = inputid
            elif 10 <= inputid < 20:
                json_file = base_path / "cc_evitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 10
            elif 20 <= inputid < 30:
                json_file = base_path / "cc_inevitable_action_yes_stories.json"
                internal_id_to_find = inputid - 20
            elif 30 <= inputid < 40:
                json_file = base_path / "cc_inevitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 30
            else:
                return "Invalid inputid for cc structure"
        elif structure == 'coc':
            if 0 <= inputid < 10:
                json_file = base_path / "coc_evitable_action_yes_stories.json"
                internal_id_to_find = inputid
            elif 10 <= inputid < 20:
                json_file = base_path / "coc_evitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 10
            elif 20 <= inputid < 30:
                json_file = base_path / "coc_inevitable_action_yes_stories.json"
                internal_id_to_find = inputid - 20
            elif 30 <= inputid < 40:
                json_file = base_path / "coc_inevitable_prevention_no_stories.json"
                internal_id_to_find = inputid - 30
            else:
                return "Invalid inputid for coc structure"
        else:
            return "Invalid structure"
        
        # Load the JSON file, which contains a list of JSON objects, and find the entry with the matching id
        with open(json_file, 'r') as f:
            inp_jsons = json.load(f)
        for j in inp_jsons:
            if j.get('id') == internal_id_to_find:
                return j, json_file.name, internal_id_to_find
        return "No matching entry found in JSON"
    else:
        return "Invalid strength"
    

# create a new df which is a copy of exp1_matched_df but with an additional column called 'json_info' which contains the output of the map_inputid_to_json_info function for each row, using the strength, structure, and exp1_input_id values from that row as inputs to the function. This will allow us to connect each unique stimulus in exp1 to the correct annotator-fed input scenario and its corresponding annotator-generated events and C/I/K labels.
exp1_matchwithannotator_df = exp1_matched_df.copy()
json_content = []
json_file_names = []
ids_to_find = []
for idx, row in exp1_matchwithannotator_df.iterrows():
    strength = row['strength']
    structure = row['structure']
    exp1_input_id = idx  # exp1_input_id is the index of the row in exp1_matchwithannotator_df
    returnval = map_inputid_to_json_info(strength, structure, exp1_input_id)
    # print(len(returnval))  # debug print to check the output of the mapping function
    content, json_file_name, internal_id_to_find = returnval
    json_content.append(content)
    json_file_names.append(json_file_name)
    ids_to_find.append(internal_id_to_find)
exp1_matchwithannotator_df['json_info'] = json_content
exp1_matchwithannotator_df['json_file_name'] = json_file_names
exp1_matchwithannotator_df['internal_id_to_find'] = ids_to_find
# preview the final df with the json info
display(HTML(exp1_matchwithannotator_df[0:2].to_html()))

,type,target,strength,structure,background,avg_likert_rating,n_likert_ratings,matched_scenario_info,json_info,json_file_name,internal_id_to_find
exp1_input_id,,,,,,,,,,,
0,good,The new bike lane system encourages residents to cycle more.,mild,coc,Maya has the opportunity to implement a new bike lane system throughout the city.,6.05,20,"{0: 'Maya, an Architect, faces a moral dilemma. Maya has the opportunity to implement a new bike lane system throughout the city. The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city. If Maya does not implement the bike lane system, the increase in parking prices would not happen. Maya implements the new bike lane system.'}","{'id': 0, 'text': 'I, an Architect, face a moral dilemma. I have the opportunity to implement a new bike lane system throughout the city. The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city. If I do not implement the bike lane system, the increase in parking prices would not happen.', 'options': {'1': 'implement the new bike lane system.', '2': 'not implement the new bike lane system.'}}",coc_evitable_action_yes_stories.json,0
1,harm,"This implementation requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days.",severe,cc,Alejandro has the opportunity to implement a new power grid system in the city.,2.10,20,"{11: 'Alejandro, an Engineer, faces a moral dilemma. Alejandro has the opportunity to implement a new power grid system in the city. As a means to achieve the significant reduction in energy consumption and energy taxes for residents, as well as promoting stable and sustainable power supply for the future, implementing a new power grid system requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days. If Alejandro does not implement the new power grid system, the city-wide blackout for several days would not happen. Alejandro implements the new power grid system.'}","{'id': 1, 'text': 'I, an Engineer, face a moral dilemma. I have the opportunity to implement a new power grid system in the city. As a means to achieve the significant reduction in energy consumption and energy taxes for residents, as well as promoting stable and sustainable power supply for the future, implementing a new power grid system requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days. If I do not implement the new power grid system, the city-wide blackout for several days would not happen.', 'options': {'1': 'implement the new power grid system.', '2': 'not implement the new power grid system.'}}",cc_evitable_action_yes_stories.json,1


In [23]:
# create a new df which is a copy of exp1_matchwithannotator_df but with only the 'exp1_input_id', 'avg_likert_rating', 'mathed_scenario_info', 'json_info', 'json_file_name', and 'internal_id_to_find' columns. 'exp1_input_id' is just the alias for the index column and therefore does not get recognized as a column field. This will allow us to focus on the relevant information for each unique stimulus in exp1 and its corresponding matched scenario and annotator-generated events and C/I/K labels.
exp1_matchedcleaned_df = exp1_matchwithannotator_df
exp1_matchedcleaned_df = exp1_matchedcleaned_df[['avg_likert_rating', 'strength', 'matched_scenario_info', 'json_info', 'json_file_name', 'internal_id_to_find']]
# preview the cleaned df with only the relevant columns
display(HTML(exp1_matchedcleaned_df[0:2].to_html()))

,avg_likert_rating,strength,matched_scenario_info,json_info,json_file_name,internal_id_to_find
exp1_input_id,,,,,,
0,6.05,mild,"{0: 'Maya, an Architect, faces a moral dilemma. Maya has the opportunity to implement a new bike lane system throughout the city. The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city. If Maya does not implement the bike lane system, the increase in parking prices would not happen. Maya implements the new bike lane system.'}","{'id': 0, 'text': 'I, an Architect, face a moral dilemma. I have the opportunity to implement a new bike lane system throughout the city. The new bike lane system encourages residents to cycle more. As a side effect of encouraging residents to cycle more, the implementation of the bike lanes increases the prices for parking cars in the city. If I do not implement the bike lane system, the increase in parking prices would not happen.', 'options': {'1': 'implement the new bike lane system.', '2': 'not implement the new bike lane system.'}}",coc_evitable_action_yes_stories.json,0
1,2.10,severe,"{11: 'Alejandro, an Engineer, faces a moral dilemma. Alejandro has the opportunity to implement a new power grid system in the city. As a means to achieve the significant reduction in energy consumption and energy taxes for residents, as well as promoting stable and sustainable power supply for the future, implementing a new power grid system requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days. If Alejandro does not implement the new power grid system, the city-wide blackout for several days would not happen. Alejandro implements the new power grid system.'}","{'id': 1, 'text': 'I, an Engineer, face a moral dilemma. I have the opportunity to implement a new power grid system in the city. As a means to achieve the significant reduction in energy consumption and energy taxes for residents, as well as promoting stable and sustainable power supply for the future, implementing a new power grid system requires a temporary shutdown of the city's power supply, causing a city-wide blackout for several days. If I do not implement the new power grid system, the city-wide blackout for several days would not happen.', 'options': {'1': 'implement the new power grid system.', '2': 'not implement the new power grid system.'}}",cc_evitable_action_yes_stories.json,1


In [24]:
# now pull out the annotator-generated events and their C/I/K labels for each scenario for the exp1_matchedcleaned_df. Loop through the df, and look at the 'json_file_name' column. There will be a folder that has the same name which lives in inside ../../annotated_outputs/. Also extract the 'internal_id_to_find' value when looping through the df. In that folder, there will be a json file named '{internal_id_to_find}_choice_1.json' which contains the annotator-fed input scenario and the annotator-generated events and C/I/K labels for that scenario. Pull out the events and their C/I/K labels for each row in exp1_matchedcleaned_df. There will be multiple events for each sneario, so we need a inner-index column. Create a new df with the exp1_input_id, avg_likert_rating, folder name (which is the same as the json_file_name), event, C, I, K, and the inner-index column which will be a number that increments for each event within the same scenario. This will allow us to connect each unique stimulus in exp1 to the correct annotator-fed input scenario and its corresponding annotator-generated events and C/I/K labels, and also have the avg_likert_rating for that stimulus in exp1 to see how it relates to the events and their C/I/K labels.
detailed_event_data = []
for idx, row in exp1_matchedcleaned_df.iterrows():
    exp1_input_id = idx  # index of the row in exp1_matchedcleaned_df
    avg_likert_rating = row['avg_likert_rating']
    json_file_name = row['json_file_name']
    internal_id_to_find = row['internal_id_to_find']
    strength = row['strength']
    strength_folder = "conditions_mild_harm_mild_good" if strength == "mild" else "conditions_severe_harm_very_good"
    
    folder_name = json_file_name.replace('.json', '')  # get the folder name by removing .json from the json file name
    annotated_output_folder = Path().resolve() / f"../../annotated_outputs/franken/{strength_folder}/{folder_name}"
    choice_1_json_path = annotated_output_folder / f"{internal_id_to_find}_choice_1.json"
    
    # Parse the events and their C/I/K labels from the choice_1.json file
    events_with_labels = parse_events_with_labels(choice_1_json_path)
    
    # Add each event and its labels to the detailed_event_data list with the corresponding info
    inner_index = 0
    for event, cik_value in events_with_labels.items():
        c_value = cik_value[0][1]  # C polarity
        i_value = cik_value[0][3]  # I polarity
        k_value = cik_value[0][5]  # K polarity
        event_utility = cik_value[1]  # Utility value
        
        detailed_event_data.append({
            'exp1_input_id': exp1_input_id,
            'avg_likert_rating': avg_likert_rating,
            'strength': strength,
            'folder_name': folder_name,
            'event': event,
            'C': c_value,
            'I': i_value,
            'K': k_value,
            'utility': event_utility,
            'inner_index': inner_index
        })
        inner_index += 1
detailed_event_df = pd.DataFrame(detailed_event_data)
# preview the detailed event df but don't print the default index column
display(HTML(detailed_event_df.to_html(index=False)))

exp1_input_id,avg_likert_rating,strength,folder_name,event,C,I,K,utility,inner_index
0,6.05,mild,coc_evitable_action_yes_stories,I implement the new bike lane system throughout the city.,+,+,+,50,0
0,6.05,mild,coc_evitable_action_yes_stories,Residents are encouraged to cycle more,+,+,+,50,1
0,6.05,mild,coc_evitable_action_yes_stories,More residents choose to use bicycles for transportation,+,+,+,50,2
0,6.05,mild,coc_evitable_action_yes_stories,Parking prices for cars in the city increase,+,-,+,0,3
0,6.05,mild,coc_evitable_action_yes_stories,Some residents face higher costs for parking their cars,+,+,+,0,4
0,6.05,mild,coc_evitable_action_yes_stories,Traffic patterns in the city may shift due to increased cycling,+,+,+,20,5
0,6.05,mild,coc_evitable_action_yes_stories,Potential reduction in car usage within the city,+,+,+,70,6
0,6.05,mild,coc_evitable_action_yes_stories,Possible improvement in air quality due to less car emissions,+,+,-,50,7
0,6.05,mild,coc_evitable_action_yes_stories,Increased safety for cyclists due to dedicated bike lanes,+,+,+,30,8
1,2.10,severe,cc_evitable_action_yes_stories,City experiences a temporary blackout lasting several days,+,+,+,-20,0
